# Airline Loyalty — Churn & Retention Pipeline

Scores 13,191 active members (Jul 2017–Jun 2018) for disengagement risk in H2 2018.  
Outputs `final_customer_results.csv` which feeds the Streamlit dashboard directly.

**Files needed in the same directory:** `Customer_Loyalty_History.csv`, `Customer_Flight_Activity.csv`, `Calendar.csv`

In [7]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, average_precision_score,
                             precision_recall_curve, precision_score,
                             recall_score, f1_score)
from xgboost import XGBClassifier

pd.set_option('display.width', 160)
RNG = 0

clh = pd.read_csv('Customer_Loyalty_History.csv')
cfa = pd.read_csv('Customer_Flight_Activity.csv')
cal = pd.read_csv('Calendar.csv', parse_dates=['Date', 'Start of Quarter'])

print(f'cfa: {len(cfa):,} rows | clh: {len(clh):,} rows')

# De-duplicate: drop exact copies first, then sum any remaining (year, month) conflicts
keys = ['Loyalty Number', 'Year', 'Month']
n_dup_keys = cfa.duplicated(subset=keys).sum()
n_dup_full  = cfa.duplicated().sum()
print(f'Duplicate keys: {n_dup_keys:,}  (exact: {n_dup_full:,}, conflicting: {n_dup_keys - n_dup_full:,})')

cfa_cleaned = (cfa.drop_duplicates()
                  .groupby(keys, as_index=False)
                  .sum())
print(f'After dedup: {len(cfa_cleaned):,} rows')

clh_cleaned = clh.copy()
n_neg = (clh_cleaned['Salary'] < 0).sum()
clh_cleaned['Salary'] = clh_cleaned['Salary'].abs()
clh_cleaned = clh_cleaned.drop(columns=['Country'])   # constant field across all rows
print(f'Negative salaries corrected: {n_neg}')

cfa_cleaned['ym'] = cfa_cleaned['Year'] * 12 + cfa_cleaned['Month']
clh_cleaned['cancel_ym'] = (clh_cleaned['Cancellation Year'] * 12
                             + clh_cleaned['Cancellation Month'])

cfa: 392,936 rows | clh: 16,737 rows
Duplicate keys: 3,871  (exact: 1,922, conflicting: 1,949)
After dedup: 389,065 rows
Negative salaries corrected: 20


In [8]:
# Prediction point T = end of June 2018.
# Population: flew at least once Jul 2017–Jun 2018 AND not cancelled by June 2018.
# Churn: no flights in H2 2018 OR membership cancelled in H2 2018.
T        = 2018 * 12 + 6
END_2018 = 2018 * 12 + 12

recent12       = cfa_cleaned[(cfa_cleaned['ym'] >= T - 11) & (cfa_cleaned['ym'] <= T)]
recent_flights = recent12.groupby('Loyalty Number')['Total Flights'].sum()
active_recent  = recent_flights[recent_flights > 0]
cancelled_by_T = set(clh_cleaned.loc[clh_cleaned['cancel_ym'] <= T, 'Loyalty Number'])
population_ids = set(active_recent.index) - cancelled_by_T

h2           = cfa_cleaned[(cfa_cleaned['ym'] > T) & (cfa_cleaned['ym'] <= END_2018)]
h2_flights   = h2.groupby('Loyalty Number')['Total Flights'].sum()
flew_h2      = set(h2_flights[h2_flights > 0].index)
cancelled_h2 = set(clh_cleaned.loc[(clh_cleaned['cancel_ym'] > T) &
                                    (clh_cleaned['cancel_ym'] <= END_2018), 'Loyalty Number'])
churned_ids  = (population_ids - flew_h2) | (population_ids & cancelled_h2)

customers = pd.DataFrame(index=pd.Index(sorted(population_ids), name='Loyalty Number'))
customers['churned'] = customers.index.isin(churned_ids).astype(int)

print(f'Population: {len(customers):,} members | '
      f'Churners: {customers["churned"].sum():,} ({customers["churned"].mean()*100:.1f}%)')

Population: 13,191 members | Churners: 422 (3.2%)


In [9]:
hist   = cfa_cleaned[cfa_cleaned['ym'] <= T]
active = hist[hist['Total Flights'] > 0]

def window(df, n):
    return df[(df['ym'] >= T - n + 1) & (df['ym'] <= T)]

def agg(df, col, how='sum'):
    return getattr(df.groupby('Loyalty Number')[col], how)().reindex(customers.index).fillna(0)

w3, w6, w12 = window(hist, 3), window(hist, 6), window(hist, 12)

# Recency
last_active = active.groupby('Loyalty Number')['ym'].max()
customers['recency_months'] = (T - last_active).reindex(customers.index).fillna(99)

# Frequency at 3m / 6m / 12m horizons
customers['flights_3m']  = agg(w3,  'Total Flights')
customers['flights_6m']  = agg(w6,  'Total Flights')
customers['flights_12m'] = agg(w12, 'Total Flights')

act3  = w3[w3['Total Flights'] > 0]
act6  = w6[w6['Total Flights'] > 0]
act12 = w12[w12['Total Flights'] > 0]
customers['active_months_3m']  = agg(act3,  'Total Flights', 'count')
customers['active_months_6m']  = agg(act6,  'Total Flights', 'count')
customers['active_months_12m'] = agg(act12, 'Total Flights', 'count')

# Momentum: recent vs prior 6m
prior6 = hist[(hist['ym'] >= T - 11) & (hist['ym'] <= T - 6)]
customers['flights_prior6']    = agg(prior6, 'Total Flights')
customers['trend_6v6']         = customers['flights_6m'] - customers['flights_prior6']
customers['ratio_6v6']         = (customers['flights_6m'] + 1) / (customers['flights_prior6'] + 1)
# 0 = activity front-loaded (fading member); 1 = all activity in last 3 months
customers['share_last3_of_12'] = customers['flights_3m'] / (customers['flights_12m'] + 1)

# Recency-weighted activity: recent months count more (0.8^k decay)
w = w12.copy()
w['wgt']      = 0.8 ** (T - w['ym'])
w['wflights'] = w['wgt'] * w['Total Flights']
customers['flights_decayed'] = agg(w, 'wflights')

# Distance & intensity
customers['distance_12m']             = agg(w12, 'Distance')
customers['distance_per_flight']      = customers['distance_12m'] / (customers['flights_12m'] + 1)
customers['avg_flights_per_active_month'] = (
    customers['flights_12m'] / customers['active_months_12m'].replace(0, np.nan)).fillna(0)

# Points engagement
customers['points_acc_12m']    = agg(w12, 'Points Accumulated')
customers['points_red_12m']    = agg(w12, 'Points Redeemed')
customers['redemption_ratio']  = customers['points_red_12m'] / (customers['points_acc_12m'] + 1)
customers['ever_redeemed_12m'] = (customers['points_red_12m'] > 0).astype(int)
red6 = agg(w6, 'Points Redeemed')
customers['redeemed_6m_flag']  = (red6 > 0).astype(int)

# Regularity: longest inactive gap and current streak
piv = w12.pivot_table(index='Loyalty Number', columns='ym',
                       values='Total Flights', aggfunc='sum', fill_value=0)
piv = (piv.reindex(columns=range(T - 11, T + 1), fill_value=0)
          .reindex(customers.index, fill_value=0))

def longest_zero_run_and_tail(row):
    longest = run = tail = 0
    for v in row:
        run = run + 1 if v == 0 else 0
        longest = max(longest, run)
    for v in row[::-1]:
        if v == 0: tail += 1
        else: break
    return longest, tail

zr = np.array([longest_zero_run_and_tail(r) for r in piv.values])
customers['max_zero_run_12m']  = zr[:, 0]
customers['zero_streak_now']   = zr[:, 1]
customers['flight_std_12m']    = piv.std(axis=1)
customers['flight_cv_12m']     = piv.std(axis=1) / (piv.mean(axis=1) + 1)
top3 = np.sort(piv.values, axis=1)[:, -3:].sum(axis=1)
customers['top3_month_share']  = top3 / (piv.sum(axis=1).values + 1)

# Distance and points deceleration
customers['flights_1m']   = agg(window(hist, 1), 'Total Flights')
d6 = agg(w6, 'Distance')
customers['dist_ratio_6v6'] = (d6 + 1) / ((customers['distance_12m'] - d6) + 1)
p6 = agg(w6, 'Points Accumulated')
customers['pts_ratio_6v6']  = (p6 + 1) / ((customers['points_acc_12m'] - p6) + 1)

# Seasonal features (Calendar.csv provides quarter mapping)
# Peak travel months in Canada: summer Jun–Aug, December holiday
cal['Year']    = cal['Date'].dt.year
cal['Month']   = cal['Date'].dt.month
cal['quarter'] = ((cal['Start of Quarter'].dt.month - 1) // 3) + 1
month_q = cal.drop_duplicates(subset=['Year', 'Month'])[['Year', 'Month', 'quarter']]

PEAK_MONTHS = {6, 7, 8, 12}
w12_peak = w12[w12['Month'].isin(PEAK_MONTHS)]
customers['peak_flights_12m']  = agg(w12_peak, 'Total Flights')
customers['peak_season_share'] = customers['peak_flights_12m'] / (customers['flights_12m'] + 1)

w12_q = w12.merge(month_q, on=['Year', 'Month'], how='left')
q_piv = (w12_q.groupby(['Loyalty Number', 'quarter'])['Total Flights']
               .sum().unstack(fill_value=0)
               .reindex(customers.index, fill_value=0))
customers['quarter_concentration'] = q_piv.max(axis=1) / (q_piv.sum(axis=1) + 1)

# Tenure
clh_idx   = clh_cleaned.set_index('Loyalty Number')
enroll_ym = clh_idx['Enrollment Year'] * 12 + clh_idx['Enrollment Month']
customers['tenure_months'] = (T - enroll_ym).reindex(customers.index)

# Demographics (one-hot; CLV and Salary excluded to prevent leakage)
demo    = clh_idx.reindex(customers.index)[['Loyalty Card', 'Education',
                                             'Marital Status', 'Gender', 'Enrollment Type']]
demo_oh = pd.get_dummies(demo, drop_first=True, dtype=int)
customers = customers.join(demo_oh)

print(f'Feature matrix: {customers.shape[0]:,} × {customers.shape[1]-1} features')
assert customers.drop(columns='churned').isnull().sum().sum() == 0, 'NaN check failed'
print('NaN check passed')

Feature matrix: 13,191 × 42 features
NaN check passed


In [10]:
y = customers['churned']
X = customers.drop(columns=['churned'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RNG)
print(f'Train: {len(X_train):,} ({y_train.sum()} churners) | '
      f'Test:  {len(X_test):,} ({y_test.sum()} churners)')

spw = (y_train == 0).sum() / (y_train == 1).sum()

model = XGBClassifier(
    n_estimators=600, max_depth=3, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.7, min_child_weight=8,
    gamma=1, reg_lambda=2, scale_pos_weight=spw,
    eval_metric='aucpr', random_state=RNG, n_jobs=-1)

# 5-fold CV on the training set. OOF probabilities are used for threshold selection
# so no test-set information leaks into the threshold choice.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG)
oof_proba = cross_val_predict(model, X_train, y_train, cv=cv,
                               method='predict_proba', n_jobs=-1)[:, 1]
cv_auc = roc_auc_score(y_train, oof_proba)
cv_pr  = average_precision_score(y_train, oof_proba)
print(f'\n5-fold CV (OOF): ROC AUC = {cv_auc:.3f} | PR AUC = {cv_pr:.3f}  '
      f'(no-skill baseline = {y_train.mean():.3f})')

prec, rec, thr = precision_recall_curve(y_train, oof_proba)
f1_scores = 2 * prec * rec / (prec + rec + 1e-9)
f2_scores = 5 * prec * rec / (4 * prec + rec + 1e-9)   # recall weighted 2× precision
t_f1 = thr[np.argmax(f1_scores[:-1])]
t_f2 = thr[np.argmax(f2_scores[:-1])]

print('\nThreshold sweep (OOF):')
print(f'{"thr":>6} {"precision":>10} {"recall":>8} {"F1":>7} {"flagged":>9}')
for t in [0.30, 0.40, 0.50, 0.60, t_f1, t_f2, 0.70, 0.80]:
    p = (oof_proba >= t).astype(int)
    print(f'{t:6.2f} {precision_score(y_train, p, zero_division=0):10.3f} '
          f'{recall_score(y_train, p):8.3f} '
          f'{f1_score(y_train, p, zero_division=0):7.3f} {p.sum():9,}')
print(f'\nbest-F1 threshold: {t_f1:.3f} | best-F2 (recall-weighted): {t_f2:.3f}')

model.fit(X_train, y_train)
proba_test = model.predict_proba(X_test)[:, 1]

print(f'\nHeld-out test set:')
print(f'ROC AUC = {roc_auc_score(y_test, proba_test):.3f} | '
      f'PR AUC = {average_precision_score(y_test, proba_test):.3f} '
      f'(baseline = {y_test.mean():.3f})')

for name, t in [('default 0.50', 0.50), (f'best-F1 {t_f1:.2f}', t_f1),
                (f'best-F2 {t_f2:.2f}', t_f2)]:
    preds = (proba_test >= t).astype(int)
    print(f'\nthreshold {name}:')
    print(confusion_matrix(y_test, preds))
    print(classification_report(y_test, preds, digits=3,
                                target_names=['retained', 'churned']))

dec = pd.DataFrame({'proba': proba_test, 'churned': y_test.values})
dec['decile'] = pd.qcut(dec['proba'].rank(method='first'), 10,
                        labels=range(10, 0, -1)).astype(int)
tab = dec.groupby('decile').agg(n=('churned', 'size'), churners=('churned', 'sum'))
tab = tab.sort_index()
tab['churn_rate']  = (tab['churners'] / tab['n']).round(3)
tab['cum_capture'] = (tab['churners'].cumsum() / dec['churned'].sum()).round(3)
print('\nCapture by decile (test set):')
print(tab.to_string())
print(f'\nTop 10% riskiest: {tab.iloc[0]["cum_capture"]*100:.0f}% of all churners captured; '
      f'top 20%: {tab.iloc[:2]["churners"].sum()/dec["churned"].sum()*100:.0f}%.')

imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print('\nTop 15 features:')
print(imp.head(15).round(4).to_string())

Train: 9,893 (316 churners) | Test:  3,298 (106 churners)

5-fold CV (OOF): ROC AUC = 0.683 | PR AUC = 0.098  (no-skill baseline = 0.032)

Threshold sweep (OOF):
   thr  precision   recall      F1   flagged
  0.30      0.060    0.557   0.108     2,956
  0.40      0.080    0.462   0.137     1,814
  0.50      0.103    0.396   0.164     1,212
  0.60      0.126    0.342   0.184       855
  0.65      0.139    0.316   0.193       719
  0.57      0.122    0.370   0.183       961
  0.70      0.140    0.247   0.179       556
  0.80      0.141    0.111   0.124       248

best-F1 threshold: 0.649 | best-F2 (recall-weighted): 0.566

Held-out test set:
ROC AUC = 0.759 | PR AUC = 0.163 (baseline = 0.032)

threshold default 0.50:
[[2771  421]
 [  46   60]]
              precision    recall  f1-score   support

    retained      0.984     0.868     0.922      3192
     churned      0.125     0.566     0.204       106

    accuracy                          0.858      3298
   macro avg      0.554     0.

In [11]:
# Refit on all labeled data; 5-seed ensemble reduces run-to-run variance
seed_probas = []
for s in range(5):
    params = model.get_params()
    params['random_state'] = s
    fm = XGBClassifier(**params)
    fm.fit(X, y)
    seed_probas.append(fm.predict_proba(X)[:, 1])
customers['churn_probability'] = np.mean(seed_probas, axis=0)

# Risk bands: High = above best-F1 threshold; Medium between 60% of F2 and F1 threshold
def risk_band(p):
    if p >= t_f1:        return 'High'
    if p >= t_f2 * 0.6: return 'Medium'
    return 'Low'
customers['risk_band']    = customers['churn_probability'].apply(risk_band)
customers['CLV']          = clh_idx['CLV'].reindex(customers.index)
customers['loyalty_card'] = clh_idx['Loyalty Card'].reindex(customers.index)

clv_thresh = customers['CLV'].quantile(0.70)   # top 30% by lifetime value = "high value"

def segment(row):
    hv = row['CLV'] >= clv_thresh
    if row['risk_band'] == 'High' and hv:   return 'VIP at risk'
    if row['risk_band'] == 'High':          return 'At risk'
    if row['risk_band'] == 'Medium' and hv: return 'High-value, cooling'
    if row['risk_band'] == 'Medium':        return 'Cooling'
    if hv:                                  return 'High-value, healthy'
    return 'Healthy'
customers['segment'] = customers.apply(segment, axis=1)

ACTIONS = {
    'VIP at risk':         'Week 1: personal call from loyalty desk + targeted offer '
                           '(companion voucher or status extension). Owner: retention team.',
    'At risk':             'Automated win-back email within 7 days: bonus-points offer '
                           'valid 60 days, route suggestions from past trips.',
    'High-value, cooling': 'Monthly: lounge pass or seat-upgrade voucher to re-engage '
                           'before recency grows. Owner: CRM automation.',
    'Cooling':             'Quarterly re-engagement email + reminder of unspent points.',
    'High-value, healthy': 'Do not discount. Surprise-and-delight touchpoint 1x/quarter.',
    'Healthy':             'Standard newsletter cadence. No spend.',
}
customers['recommended_action'] = customers['segment'].map(ACTIONS)

print(customers.groupby('segment').agg(
    n=('churned', 'size'),
    actual_churn_rate=('churned', 'mean'),
    avg_probability=('churn_probability', 'mean'),
    avg_CLV=('CLV', 'mean')).round(3).to_string())

out = customers.reset_index()
out.to_csv('final_customer_results.csv', index=False)
print(f'\nSaved {len(out):,} customers → final_customer_results.csv')

                        n  actual_churn_rate  avg_probability    avg_CLV
segment                                                                 
At risk               826              0.228            0.787   4919.937
Cooling              1936              0.054            0.443   4845.887
Healthy              6470              0.000            0.185   4826.716
High-value, cooling   825              0.046            0.439  15466.293
High-value, healthy  2798              0.001            0.185  15145.134
VIP at risk           336              0.262            0.792  15340.222

Saved 13,191 customers → final_customer_results.csv


In [12]:
%%writefile retention_app.py
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Retention Control Tower",
                   page_icon="\U0001f6eb", layout="wide")

st.markdown("""
<style>
  .block-container {padding-top: 2.2rem;}
  h1 {font-weight: 800; letter-spacing: -0.5px;}
  [data-testid="stMetricValue"] {font-size: 1.9rem;}
  .seg-card {border: 1px solid #d9d9d9; border-left: 6px solid #888;
             border-radius: 8px; padding: 14px 18px; margin-bottom: 12px;
             background: #fafafa;}
  .seg-card h4 {margin: 0 0 4px 0;}
  .seg-card .meta {color: #666; font-size: 0.85rem; margin-bottom: 6px;}
  .risk-high {border-left-color: #c62828;}
  .risk-med  {border-left-color: #ef6c00;}
  .risk-low  {border-left-color: #2e7d32;}
</style>
""", unsafe_allow_html=True)

@st.cache_data
def load_data():
    df = pd.read_csv("final_customer_results.csv")
    df["Loyalty Number"] = df["Loyalty Number"].astype(int)
    return df

try:
    df = load_data()
except FileNotFoundError:
    st.error("final_customer_results.csv not found. "
             "Run Final_pipeline.ipynb first, then place the CSV next to this file.")
    st.stop()

SEG_RISK = {"VIP at risk": "risk-high", "At risk": "risk-high",
            "High-value, cooling": "risk-med", "Cooling": "risk-med",
            "High-value, healthy": "risk-low", "Healthy": "risk-low"}
SEG_ORDER = list(SEG_RISK.keys())

st.title("Retention Control Tower")
st.caption("Scored at end of June 2018 · predicts disengagement risk for "
           "July–December 2018 · refresh by re-running Final_pipeline.ipynb")

at_risk    = df[df["risk_band"] == "High"]
vip_risk   = df[df["segment"] == "VIP at risk"]
clv_exposed = at_risk["CLV"].sum()

c1, c2, c3, c4 = st.columns(4)
c1.metric("Active members",   f"{len(df):,}")
c2.metric("Flagged high risk", f"{len(at_risk):,}",
          help="Members above the high-risk threshold. "
               "~1 in 5 of these disengage within 6 months vs ~1 in 30 overall.")
c3.metric("VIPs at risk",     f"{len(vip_risk):,}",
          help="High risk AND top-30% lifetime value. Prioritise these first.")
c4.metric("CLV exposed",      f"${clv_exposed/1e6:,.1f}M",
          help="Combined lifetime value of all high-risk members.")

st.divider()

tab_list, tab_play, tab_member = st.tabs(
    ["\U0001f4cb Action list", "\U0001f5fa\ufe0f Segment playbook", "\U0001f50e Member lookup"])

with tab_list:
    st.subheader("Who needs attention first")
    st.caption("Sorted by risk. Filter, review, download — the CSV is ready to hand to the ops team.")

    f1, f2, f3 = st.columns([2, 2, 3])
    seg_pick  = f1.multiselect("Segments", SEG_ORDER,
                               default=["VIP at risk", "At risk"])
    card_pick = f2.multiselect("Loyalty card",
                               sorted(df["loyalty_card"].unique()),
                               default=sorted(df["loyalty_card"].unique()))
    top_n     = f3.slider("How many members can the team contact?",
                          50, 3000, 500, step=50)

    view = (df[df["segment"].isin(seg_pick) & df["loyalty_card"].isin(card_pick)]
              .sort_values("churn_probability", ascending=False)
              .head(top_n))

    if len(view) == 0:
        st.info("No members match these filters. Widen the segment or card selection.")
    else:
        exp_churn = view["churn_probability"].sum()
        st.markdown(f"**{len(view):,} members selected** · model expects roughly "
                    f"**{exp_churn:,.0f}** to disengage if nothing is done · "
                    f"combined CLV **${view['CLV'].sum():,.0f}**")

        show = view[["Loyalty Number", "segment", "churn_probability",
                     "CLV", "loyalty_card", "recency_months", "flights_12m",
                     "zero_streak_now", "recommended_action"]].rename(columns={
            "churn_probability":  "Risk score",
            "segment":            "Segment",
            "loyalty_card":       "Card",
            "recency_months":     "Months since last flight",
            "flights_12m":        "Flights (12m)",
            "zero_streak_now":    "Inactive streak (months)",
            "recommended_action": "Next action"})
        st.dataframe(
            show, use_container_width=True, hide_index=True, height=430,
            column_config={
                "Risk score": st.column_config.ProgressColumn(
                    "Risk score", min_value=0, max_value=1, format="%.2f"),
                "CLV": st.column_config.NumberColumn(format="$%d")})

        st.download_button(
            "Download this contact list (CSV)",
            show.to_csv(index=False).encode(),
            file_name="retention_contact_list.csv", mime="text/csv")

with tab_play:
    st.subheader("One play per segment")
    st.caption("Six segments from two questions: how likely to disengage (model risk) "
               "and how valuable (top 30% CLV = high). Each card is the standing "
               "instruction for that group.")

    stats = df.groupby("segment").agg(
        n=("Loyalty Number", "size"),
        avg_risk=("churn_probability", "mean"),
        churn_rate=("churned", "mean"),
        clv=("CLV", "mean"),
        action=("recommended_action", "first"))

    left, right = st.columns(2)
    for i, seg in enumerate(SEG_ORDER):
        if seg not in stats.index:
            continue
        r      = stats.loc[seg]
        target = left if i % 2 == 0 else right
        target.markdown(f"""
<div class="seg-card {SEG_RISK[seg]}">
  <h4>{seg}</h4>
  <div class="meta">{int(r['n']):,} members · observed disengagement
  {r['churn_rate']*100:.0f}% · avg lifetime value ${r['clv']:,.0f}</div>
  <div><b>Play:</b> {r['action']}</div>
</div>""", unsafe_allow_html=True)

    st.divider()
    st.markdown("**Where the value sits**")
    pivot = df.pivot_table(
        index="risk_band",
        columns=df["CLV"] >= df["CLV"].quantile(0.70),
        values="CLV", aggfunc="sum").rename(
        columns={False: "Standard value", True: "High value (top 30%)"})
    pivot = pivot.reindex(["High", "Medium", "Low"]) / 1e6
    st.bar_chart(pivot, height=260, use_container_width=True)
    st.caption("Total lifetime value ($M) by risk band. "
               "High-value bars in High/Medium risk = immediate revenue at stake.")

with tab_member:
    st.subheader("Look up one member")
    q = st.text_input("Loyalty number", placeholder="e.g. 100018")
    if q:
        try:
            row = df[df["Loyalty Number"] == int(q)]
        except ValueError:
            row = df.iloc[0:0]
        if len(row) == 0:
            st.warning("Member not found. Members with no flights in the last 12 months "
                       "are outside the scored population.")
        else:
            r = row.iloc[0]
            a, b, c, d = st.columns(4)
            a.metric("Risk score",      f"{r['churn_probability']:.2f}")
            b.metric("Segment",         r["segment"])
            c.metric("Lifetime value",  f"${r['CLV']:,.0f}")
            d.metric("Card",            r["loyalty_card"])
            e, f, g, h = st.columns(4)
            e.metric("Flights, last 12m",       int(r["flights_12m"]))
            f.metric("Months since last flight", int(r["recency_months"]))
            g.metric("Inactive streak now",      f"{int(r['zero_streak_now'])} mo")
            h.metric("Tenure",                   f"{int(r['tenure_months'])} mo")
            st.markdown(f"**Recommended next action:** {r['recommended_action']}")

Overwriting retention_app.py


Exception ignored in: <function ResourceTracker.__del__ at 0x1086e9bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x104b65bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x105969bc0>
Traceback (most recent call last